# 03_optimizacion — reasignación presupuestal (Mapa 3 de la app)

Toma `indice_priorizacion_tau` de `app_maestro_distrital.csv` y redistribuye el **mismo presupuesto total** entre distritos, con la restricción de que ninguno pierda más de un % políticamente viable de lo que ya tiene. Resultado: `OPTIMIZACION_FINAL.csv`.

**Dos avisos importantes — léase antes de correr esto en la presentación.**

### Aviso 1 — se encontró un problema de datos nuevo, no detectado antes

Al construir el presupuesto "actual" para esta optimización, `gasto_anemia_pan` (el gasto del PAN, programa 0001) sale en **0 exacto para el 100% de los distritos en 2023, 2024 y 2025** — no un porcentaje alto, el 100%. `gasto_total` (todo el gasto del distrito, sin filtrar por programa) es normal y creciente esos mismos años, así que **no es que el Estado dejó de gastar en nutrición — es que la extracción de SIAF filtrada al programa 0001 dejó de encontrar coincidencias desde 2023**, probablemente por un cambio de código o clasificación presupuestal que `00_prep_dataset.ipynb` no contempla todavía.

Esto afecta más que esta optimización: **`01_spending_model`, `02_causal_forest` y `02_contrafractual` entrenaron con un T que tiene 3 de 5 años estructuralmente en cero por este motivo, no solo por falta real de ejecución.** No se puede arreglar aquí sin tocar `00_prep_dataset.ipynb`, y no se hizo porque no se pidió explícitamente. Vale la pena que lo sepas para la siguiente iteración del proyecto — hoy toca trabajar con lo que hay.

**Para esta optimización específica**, el presupuesto "actual" se calcula solo con **2021 y 2022** — los únicos años donde el dato es real.

### Aviso 2 — qué significa "optimizar" aquí, y qué NO significa

Esta optimización **no calcula "casos de anemia evitados"** como número — hacerlo requeriría tomar el signo y la magnitud de τ al pie de la letra como un efecto causal real, y ya se estableció en `02_causal_forest.ipynb` que el ATE salió positivo (más gasto asociado a más anemia, por focalización, no causalidad limpia). Si se optimizara literalmente para "minimizar anemia predicha" con τ tal cual, el resultado sería recomendar **cortar** el gasto en todos lados — lo opuesto a la intención del proyecto, y potencialmente dañino si alguien lo tomara en serio.

Lo que sí se hace, de forma defendible: usar τ como **peso de prioridad relativa entre distritos** — el presupuesto se mueve hacia los distritos con τ más alto (donde el patrón gasto-anemia es más fuerte, según tu propio índice de priorización), y se aleja de los de τ más bajo, dentro del mismo total y sin dejar a nadie en una pérdida drástica. Es optimización matemática real (programación lineal con restricciones), pero el resultado se comunica como **"hacia dónde priorizar"**, nunca como "esto evita X casos de anemia".

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import linprog

df = pd.read_csv("data/clean/merged/causal_model_data.csv")
maestro = pd.read_csv("data/predictions/app_maestro_distrital.csv")

print(f"Distritos en app_maestro_distrital: {len(maestro)}")


## 1. Presupuesto base (solo 2021-2022, por el Aviso 1)

Se toma el año más reciente **entre 2021 y 2022** con `t_no_disponible=0` y `outlier_administrativo=0` para cada distrito. Si un distrito no ejecutó nada del PAN en ninguno de esos 2 años, su base queda en 0 (caso real de no-ejecución, distinto del bug de 2023-2025).

In [ ]:
validT = df[
    (df["anio"].isin([2021, 2022])) & (df["t_no_disponible"] == 0) & (df["outlier_administrativo"] == 0)
].sort_values("anio")

gasto_ref = validT.groupby("ubigeo").tail(1)[["ubigeo", "anio", "gasto_anemia_pan"]]
gasto_ref = gasto_ref.rename(columns={"anio": "anio_referencia_gasto_opt", "gasto_anemia_pan": "gasto_actual_soles"})

base = maestro[[
    "ubigeo", "distrito", "provincia", "departamento",
    "indice_priorizacion_tau", "tau_es_confiable", "dato_priorizacion_disponible",
]].merge(gasto_ref, on="ubigeo", how="left")

base["gasto_actual_soles"] = base["gasto_actual_soles"].fillna(0.0)
base["anio_referencia_gasto_opt"] = base["anio_referencia_gasto_opt"].fillna(0).astype(int)

print(f"Presupuesto total nacional (base 2021-2022): S/ {base['gasto_actual_soles'].sum():,.0f}")
print(f"Distritos sin ejecución en 2021-2022: {(base['gasto_actual_soles']==0).sum()}")


## 2. Quiénes entran a la optimización

Solo los distritos con `indice_priorizacion_tau` disponible (1,876 de 1,889) — sin τ no hay base para decidir si suben o bajan, así que los 13 restantes se quedan con su presupuesto sin cambios, fuera del problema de optimización.

In [ ]:
incluidos = base[base["dato_priorizacion_disponible"] == True].copy().reset_index(drop=True)
excluidos = base[base["dato_priorizacion_disponible"] == False].copy().reset_index(drop=True)

print(f"Incluidos en la optimización (con τ): {len(incluidos)}")
print(f"Excluidos (sin τ, presupuesto sin cambios): {len(excluidos)}")


## 3. El problema de optimización

**Maximizar** Σ (τᵢ × gasto_nuevoᵢ) — mover el presupuesto hacia τ alto.

**Sujeto a:**
- Σ gasto_nuevoᵢ = Σ gasto_actualᵢ (mismo presupuesto total, entre los incluidos)
- gasto_nuevoᵢ ≥ gasto_actualᵢ × (1 − 20%) — ningún distrito pierde más del 20% de lo que ya tenía
- gasto_nuevoᵢ ≤ gasto_actualᵢ + 20% del promedio nacional — techo de subida. Se usa un techo **aditivo** (no multiplicativo) porque 680 distritos parten de S/0, y un límite de "+20% de lo actual" los dejaría atrapados en cero para siempre (20% de 0 es 0). El techo aditivo les da espacio real para recibir fondos.

In [ ]:
MAX_PERDIDA_PCT = 0.20

gasto_actual = incluidos["gasto_actual_soles"].values
tau = incluidos["indice_priorizacion_tau"].values
n = len(incluidos)

promedio_nacional = gasto_actual.mean()
piso = np.maximum(0, gasto_actual * (1 - MAX_PERDIDA_PCT))
techo = gasto_actual + MAX_PERDIDA_PCT * promedio_nacional

c = -tau  # linprog minimiza, así que se usa -tau para maximizar tau*x
A_eq = np.ones((1, n))
b_eq = [gasto_actual.sum()]
bounds = list(zip(piso, techo))

resultado = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
print(f"Optimización exitosa: {resultado.success} — {resultado.message}")

gasto_nuevo = resultado.x
print(f"Presupuesto total antes:  S/ {gasto_actual.sum():,.2f}")
print(f"Presupuesto total después: S/ {gasto_nuevo.sum():,.2f}")
print(f"Diferencia (debe ser ~0):  S/ {gasto_nuevo.sum() - gasto_actual.sum():,.6f}")


## 4. Validación

Chequeo de que ningún distrito quedó fuera de sus límites (piso/techo) y que el presupuesto total se conservó exactamente.

In [ ]:
incluidos["gasto_propuesto_soles"] = gasto_nuevo
incluidos["cambio_soles"] = incluidos["gasto_propuesto_soles"] - incluidos["gasto_actual_soles"]
incluidos["cambio_porcentual"] = np.where(
    incluidos["gasto_actual_soles"] > 0,
    incluidos["cambio_soles"] / incluidos["gasto_actual_soles"],
    np.nan
)

viol_piso = (gasto_nuevo < piso - 1).sum()
viol_techo = (gasto_nuevo > techo + 1).sum()
print(f"Violaciones de piso (tolerancia S/1): {viol_piso}")
print(f"Violaciones de techo (tolerancia S/1): {viol_techo}")
assert viol_piso == 0 and viol_techo == 0, "hay distritos fuera de sus límites, revisar"
assert abs(gasto_nuevo.sum() - gasto_actual.sum()) < 1, "el presupuesto total no se conservó"
print("✅ Presupuesto conservado y todos los distritos dentro de sus límites")

print()
print("Top 10 distritos que MÁS ganan:")
print(incluidos.sort_values("cambio_soles", ascending=False)
      [["distrito","departamento","indice_priorizacion_tau","gasto_actual_soles","gasto_propuesto_soles","cambio_soles"]]
      .head(10).to_string(index=False))
print()
print("Top 10 distritos que MÁS pierden:")
print(incluidos.sort_values("cambio_soles")
      [["distrito","departamento","indice_priorizacion_tau","gasto_actual_soles","gasto_propuesto_soles","cambio_soles"]]
      .head(10).to_string(index=False))


## 5. Guardar `OPTIMIZACION_FINAL.csv`

Se juntan los incluidos (con su nuevo presupuesto propuesto) y los excluidos (sin cambio, marcados claramente), en un solo archivo con nombres de columna sin ambigüedad.

In [ ]:
excluidos["gasto_propuesto_soles"] = excluidos["gasto_actual_soles"]
excluidos["cambio_soles"] = 0.0
excluidos["cambio_porcentual"] = np.nan

final = pd.concat([incluidos, excluidos], ignore_index=True)
final["incluido_en_optimizacion"] = final["dato_priorizacion_disponible"]

final = final.rename(columns={
    "anio_referencia_gasto_opt": "anio_referencia_presupuesto_base",
})

col_order = [
    "ubigeo", "distrito", "provincia", "departamento",
    "gasto_actual_soles", "anio_referencia_presupuesto_base",
    "indice_priorizacion_tau", "tau_es_confiable", "incluido_en_optimizacion",
    "gasto_propuesto_soles", "cambio_soles", "cambio_porcentual",
]
final = final[col_order].sort_values("cambio_soles", ascending=False).reset_index(drop=True)

OUTPUT_DIR = Path("data/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final.to_csv(OUTPUT_DIR / "OPTIMIZACION_FINAL.csv", index=False)

print(f"Guardado: {OUTPUT_DIR / 'OPTIMIZACION_FINAL.csv'}  ({len(final)} distritos)")
final.head(10)
